# Ocean Wave Analysis

Measurements from a floating buoy off Mooloolaba, Australia, taken every 30
minutes from January 2017 to June 2019.

This notebook walks through the project step by step. The actual code lives in
the `src` folder and is imported here, so the notebook tells the story and the
code files do the work. That means the same tested code produces the numbers
here, in `run_analysis.py`, and in the README.

**Steps**
1. Open the file and see what is in it
2. Problem 1: the file uses two different date formats
3. Problem 2: broken readings written as -99.9
4. Question 1: how big are the waves usually?
5. Question 2: which months have the biggest waves?
6. Question 3: which direction do the waves come from?

In [1]:
import sys
sys.path.append("..")   # so we can import the src folder from here

import pandas as pd

from src import analysis, clean_data, load_data

## 1. Open the file

Notice that `load_raw_waves` does **not** try to read the dates. It reads them as
plain text for now. The reason becomes clear in the next step.

In [2]:
raw = load_data.load_raw_waves()
print(f"{len(raw):,} readings, {len(raw.columns)} columns")
raw.head()

43,728 readings, 7 columns


,Date/Time,Hs,Hmax,Tz,Tp,Peak Direction,SST
0,01/01/2017 00:00,-99.900,-99.90,-99.900,-99.900,-99.9,-99.90
1,01/01/2017 00:30,0.875,1.39,4.421,4.506,-99.9,-99.90
2,01/01/2017 01:00,0.763,1.15,4.520,5.513,49.0,25.65
3,01/01/2017 01:30,0.770,1.41,4.582,5.647,75.0,25.50
4,01/01/2017 02:00,0.747,1.16,4.515,5.083,91.0,25.45


### Is anything missing?

Python says nothing is missing.

In [3]:
raw.isna().sum()

Date/Time         0
Hs                0
Hmax              0
Tz                0
Tp                0
Peak Direction    0
SST               0
dtype: int64

That is misleading, and it is the first trap in this data.

When an instrument on the buoy breaks, it does not leave the space blank. It
writes **`-99.9`**, which looks like an ordinary number. So Python sees nothing
wrong while broken readings sit in the data waiting to be averaged.

Look at the very first reading in the file:

In [4]:
raw.head(1)

,Date/Time,Hs,Hmax,Tz,Tp,Peak Direction,SST
0,01/01/2017 00:00,-99.9,-99.9,-99.9,-99.9,-99.9,-99.9


Every measurement is `-99.9`. That is not a freezing cold day with waves
going backwards — it is a broken reading.

Here is how many are affected in each column:

In [5]:
broken_table = load_data.describe_raw_structure(raw)
broken_table[["column", "meaning", "broken_readings", "percent_broken"]]

,column,meaning,broken_readings,percent_broken
0,Hs,Wave height (m),85,0.19
1,Hmax,Biggest single wave in the reading (m),85,0.19
2,Tz,Average seconds between waves,85,0.19
3,Tp,Seconds between the biggest waves,85,0.19
4,Peak Direction,Direction the waves come from (degrees),271,0.62
5,SST,Water temperature (degrees C),262,0.60


## 2. Problem 1: the file uses two date formats

This one is worse, because nothing signals it at all.

The normal thing to do would be to let Python work the dates out automatically.
Let's see what happens.

In [6]:
automatic = pd.to_datetime(raw["Date/Time"], errors="coerce")

print(f"readings Python could not read: {automatic.isna().sum():,}")
print(f"date range: {automatic.min()}  to  {automatic.max()}")

readings Python could not read: 5,232
date range: 2017-01-01 00:00:00  to  2019-12-06 23:30:00


Two things are wrong.

**5,232 readings became unreadable** — 12% of the file, silently dropped without
a warning.

**The last date is December 6th, 2019**, but the file is only supposed to run to
**June 2019**. December 6th is June 12th with the day and month swapped.

Let's look at which readings failed:

In [7]:
failed = raw.loc[automatic.isna(), "Date/Time"]
print("first few that failed:")
print(failed.head(4).to_string(index=False))

# pull the year out of each failed date: "13/01/2019 00:00" -> "2019"
years_that_failed = failed.str.split("/").str[2].str.split(" ").str[0]
print(f"\nall the failures are in: {years_that_failed.unique()}")

first few that failed:
13/01/2019 00:00
13/01/2019 00:30
13/01/2019 01:00
13/01/2019 01:30

all the failures are in: ['2019']


Every failure is in 2019. That is the clue.

Now let's split each date into its numbers and look at them year by year.
Remember: **a month can never be more than 12, but a day can be.**

In [8]:
# "13/01/2019 04:30" -> 13, 01, 2019
date_only = raw["Date/Time"].str.split(" ").str[0]
fields = date_only.str.split("/", expand=True)

parts = pd.DataFrame({
    "first_number": fields[0].astype(int),
    "second_number": fields[1].astype(int),
    "year": fields[2].astype(int),
})

parts.groupby("year").agg(
    biggest_first_number=("first_number", "max"),
    biggest_second_number=("second_number", "max"),
    readings=("year", "size"),
)

,biggest_first_number,biggest_second_number,readings
year,,,
2017,12,31,17520
2018,12,31,17520
2019,31,6,8688


There it is.

- **2017 and 2018**: the first number never goes above 12, the second reaches 31.
  So the first number is the month → **month/day/year**
- **2019**: the first number reaches 31, the second never goes above 6.
  So the first number is the day → **day/month/year**

The file really does change format partway through. Python picked month-first
(correct for 2017 and 2018), then hit 2019 and either failed or silently swapped
the day and month.

### The fix

`detect_date_format` turns that reasoning into code, and `parse_dates` applies it
to each year separately.

In [9]:
format_table = clean_data.report_date_formats(raw)
format_table

,year,rows,detected_format,style
0,2017,17520,%m/%d/%Y %H:%M,month-first
1,2018,17520,%m/%d/%Y %H:%M,month-first
2,2019,8688,%d/%m/%Y %H:%M,day-first


In [10]:
with_dates = clean_data.parse_dates(raw)
with_dates[["Date/Time", "timestamp"]].head(3)

,Date/Time,timestamp
0,01/01/2017 00:00,2017-01-01 00:00:00
1,01/01/2017 00:30,2017-01-01 00:30:00
2,01/01/2017 01:00,2017-01-01 01:00:00


### Checking the fix actually worked

Rather than just looking at the dates and hoping, I check them against something
that has to be true: **the buoy takes a reading every 30 minutes.**

If the dates are right, every reading is exactly 30 minutes after the one before,
with no gaps and no repeats. If a format were wrong, the order would jump around
and this check would fail.

In [11]:
checks = clean_data.check_timeline(with_dates)
for name, value in checks.items():
    print(f"{name:24s} {value}")

total_rows               43728
unparsed_dates           0
duplicate_timestamps     0
gaps_not_30_minutes      0
first_reading            2017-01-01 00:00:00
last_reading             2019-06-30 23:30:00


Nothing unreadable, nothing duplicated, and all 43,727 gaps exactly 30
minutes apart — running from 1 January 2017 to 30 June 2019, which matches the
file name.

That is proof, not a guess. This same check is one of the automatic tests in
`tests/test_clean_data.py`.

## 3. Problem 2: removing the broken readings

I remove the whole reading when any column has `-99.9` in it, rather than just
that one value. The buoy tends to lose a whole reading rather than one
instrument — 82 readings have the broken code in every single column.

In [12]:
waves = clean_data.clean_waves(raw)

removed = len(raw) - len(waves)
print(f"removed  {removed} broken readings ({removed / len(raw) * 100:.2f}%)")
print(f"kept     {len(waves):,} readings ({len(waves) / len(raw) * 100:.1f}%)")

removed  274 broken readings (0.63%)
kept     43,454 readings (99.4%)


Less than 1% of the data, but it matters. Here is the effect on water
temperature:

In [13]:
pd.DataFrame({
    "broken left in": raw["SST"].describe()[["mean", "std", "min"]],
    "broken removed": waves["SST"].describe()[["mean", "std", "min"]],
}).round(2)

,broken left in,broken removed
mean,23.21,23.95
std,9.81,2.23
min,-99.90,19.80


Minus 99.9 degrees is obviously not a real ocean temperature.

Now the data can be trusted, so we can start answering questions.

## 4. Question 1: how big are the waves usually?

In [14]:
analysis.wave_height_summary(waves)

{'readings': 43454,
 'typical_m': np.float64(1.13),
 'average_m': np.float64(1.24),
 'smallest_m': np.float64(0.29),
 'biggest_m': np.float64(4.26)}

I report the **median** as the typical wave rather than the average. The
median is the middle value, so half the readings are below it and half above.
That is fairer here because a few very big days pull the average upwards.

Now, how often do the waves actually reach a decent size?

In [15]:
height_table = analysis.how_often_above_each_height(waves)
height_table

,at_least_m,readings,percent_of_time
0,0.5,42470,97.7
1,1.0,26284,60.5
2,1.5,11823,27.2
3,2.0,4113,9.5
4,2.5,1072,2.5
5,3.0,236,0.5


So it is almost never completely flat, it is over a metre about **6 days out
of 10**, and big surf is rare — over 3 m happens **half a percent** of the time.

## 5. Question 2: which months have the biggest waves?

One thing to be careful about first. The data starts in January 2017 and stops in
June 2019, so **January to June appear three times while July to December only
appear twice**. Simply averaging everything would weight the first half of the
year more heavily.

So I work it out two ways: all readings pooled, and each year counted equally.
If they agree, the pattern is real.

In [16]:
monthly = analysis.averages_by_month(waves)
monthly[["month_name", "simple_average", "fair_average", "water_temp_c", "readings"]].round(2)

,month_name,simple_average,fair_average,water_temp_c,readings
month,,,,,
1,Jan,1.17,1.14,26.47,4452
2,Feb,1.58,1.42,26.87,3890
3,Mar,1.43,1.56,26.53,4444
4,Apr,1.56,1.56,24.90,4308
5,May,1.22,1.25,23.34,4454
6,Jun,1.19,1.22,21.83,4311
7,Jul,0.86,0.86,21.04,2970
8,Aug,0.85,0.85,20.89,2964
9,Sep,0.99,0.99,21.23,2872


In [17]:
analysis.biggest_and_smallest_months(monthly)

{'biggest_month': 'Mar',
 'biggest_m': np.float64(1.56),
 'smallest_month': 'Aug',
 'smallest_m': np.float64(0.85),
 'times_bigger': np.float64(1.8)}

Both methods give the same shape, so the pattern is real: **big waves in
February to April, small waves in July and August.**

There is a nice bonus in the water temperature column. The biggest waves arrive
when the water is **warmest** (around 26–27 °C in February to April), and the
calmest months are also the coldest (around 21 °C in July and August). The best
surf comes when you need the least wetsuit.

## 6. Question 3: which direction do the waves come from?

The buoy records the direction each wave arrives from, in degrees. I group those
into the eight compass directions and count them.

In [18]:
directions = analysis.count_by_direction(waves)
directions

,direction,readings,average_height_m,percent_of_time
0,N,150,1.00,0.3
1,NE,3559,1.04,8.2
2,E,26501,1.30,61.0
3,SE,13074,1.17,30.1
4,S,138,0.75,0.3
5,SW,29,0.58,0.1
6,W,3,1.18,0.0


In [19]:
analysis.most_common_direction(directions)

{'direction': 'E',
 'percent_of_time': np.float64(61.0),
 'average_height_m': np.float64(1.3)}

Waves arrive from the **east 61% of the time**, and over **90%** once you
include the south-east. That makes sense — Mooloolaba faces east into the Coral
Sea, and the land blocks anything from the west.

The east is not just the most common direction, it also brings the **biggest**
waves.

One caution: a few directions have almost no readings behind them — west has
**3**. An average of 3 readings does not mean anything, so the chart in the
README only shows averages for directions with at least 100 readings.

---

## Summary

| Question | Answer |
|---|---|
| How big are the waves? | Typically 1.13 m; over 1 m about 60% of the time |
| Biggest month? | March (1.56 m) vs August (0.85 m) — about 1.8x |
| Where from? | The east, 61% of the time |

The most useful part was not any of the three answers. It was finding that the
file changed date format partway through, and that broken readings were written
as `-99.9` instead of being left blank. Neither showed an error message, and both
would have quietly produced wrong answers.